#Import dependencies

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

#Extract data from csv files

In [2]:
df=pd.DataFrame(pd.read_csv("/content/happiness2.csv"))
df.columns

Index(['Year', 'Rank', 'Country name', 'Ladder score', 'upperwhisker',
       'lowerwhisker', 'Explained by: Log GDP per capita',
       'Explained by: Social support', 'Explained by: Healthy life expectancy',
       'Explained by: Freedom to make life choices',
       'Explained by: Generosity', 'Explained by: Perceptions of corruption',
       'Dystopia + residual'],
      dtype='object')

In [3]:
df.columns=['Year', 'Rank', 'Country name', 'Ladder score', 'upperwhisker',
       'lowerwhisker', 'GDP',
       'Social support', 'Healthy life expectancy',
       'Freedom to make life choices',
       'Generosity', 'Perceptions of corruption',
       'Dystopia + residual']

# Sampling the data

In [4]:
print(df.head())

   Year  Rank Country name  Ladder score  upperwhisker  lowerwhisker    GDP  \
0  2024     1      Finland         7.736         7.810         7.662  1.749   
1  2024     2      Denmark         7.521         7.611         7.431  1.825   
2  2024     3      Iceland         7.515         7.606         7.425  1.799   
3  2024     4       Sweden         7.345         7.427         7.262  1.783   
4  2024     5  Netherlands         7.306         7.372         7.240  1.822   

   Social support  Healthy life expectancy  Freedom to make life choices  \
0           1.783                    0.824                         0.986   
1           1.748                    0.820                         0.955   
2           1.840                    0.873                         0.971   
3           1.698                    0.889                         0.952   
4           1.667                    0.844                         0.860   

   Generosity  Perceptions of corruption  Dystopia + residual  
0   

In [5]:
print(df.tail())

     Year  Rank              Country name  Ladder score  upperwhisker  \
870  2019   149  Central African Republic         3.476         3.702   
871  2019   150                    Rwanda         3.312         3.415   
872  2019   151                  Zimbabwe         3.299         3.414   
873  2019   152               South Sudan         2.817         3.028   
874  2019   153               Afghanistan         2.567         2.628   

     lowerwhisker    GDP  Social support  Healthy life expectancy  \
870         3.250  0.041           0.000                    0.000   
871         3.210  0.343           0.523                    0.572   
872         3.184  0.426           1.048                    0.375   
873         2.606  0.289           0.553                    0.209   
874         2.506  0.301           0.356                    0.266   

     Freedom to make life choices  Generosity  Perceptions of corruption  \
870                         0.293       0.254                      0.0

#Spliiting the dataset into training and testing sets

In [6]:
traindf=df[df["Year"]<=2022]
testdf=df[df["Year"]>2022]

features=['GDP',
       'Social support', 'Healthy life expectancy',
       'Freedom to make life choices',
       'Generosity', 'Perceptions of corruption']

xtrain=traindf[features]
ytrain=traindf['Ladder score']

xtest=testdf[features]
ytest=testdf['Ladder score']

#Checking for null values

In [7]:
print(df.isna().sum())

Year                            0
Rank                            0
Country name                    0
Ladder score                    0
upperwhisker                    0
lowerwhisker                    0
GDP                             3
Social support                  3
Healthy life expectancy         5
Freedom to make life choices    4
Generosity                      3
Perceptions of corruption       4
Dystopia + residual             7
dtype: int64


In [8]:
xtrain.isnull().sum(),xtest.isnull().sum()

(GDP                             0
 Social support                  0
 Healthy life expectancy         1
 Freedom to make life choices    0
 Generosity                      0
 Perceptions of corruption       0
 dtype: int64,
 GDP                             3
 Social support                  3
 Healthy life expectancy         4
 Freedom to make life choices    4
 Generosity                      3
 Perceptions of corruption       4
 dtype: int64)

In [9]:
ytrain.isnull().sum(),ytest.isnull().sum()

(np.int64(0), np.int64(0))

In [10]:
imputer=SimpleImputer(strategy="mean")
xtrain=imputer.fit_transform(xtrain)
xtest=imputer.transform(xtest)

xscaler = MinMaxScaler()
yscaler = MinMaxScaler()

xtrainscaled = xscaler.fit_transform(xtrain)
xtestscaled  = xscaler.transform(xtest)

ytrainscaled = yscaler.fit_transform(ytrain.values.reshape(-1, 1))
ytestscaled  = yscaler.transform(ytest.values.reshape(-1, 1))

In [11]:
print(np.isnan(xtrainscaled).sum())
print(np.isnan(xtestscaled).sum())

0
0


#Creating the model

In [12]:
model2025=Ridge(alpha=10.0)
model2025.fit(xtrainscaled,ytrainscaled)

Ridge(alpha=10.0)

#Testing the model's predictions for 2023 and 2024

In [13]:
ytestpreds=model2025.predict(xtestscaled).reshape(-1,1)
ytestpreds=yscaler.inverse_transform(ytestpreds)

In [14]:
df2=pd.DataFrame({'Country': testdf['Country name'].values,'Year': testdf['Year'].values,'Actual': ytest.values,'Predicted': ytestpreds.flatten()})
print(df2)

          Country  Year  Actual  Predicted
0         Finland  2024   7.736   7.795816
1         Denmark  2024   7.521   7.762057
2         Iceland  2024   7.515   7.614133
3          Sweden  2024   7.345   7.759749
4     Netherlands  2024   7.306   7.456008
..            ...   ...     ...        ...
285      DR Congo  2023   3.295   4.518765
286  Sierra Leone  2023   3.245   4.472041
287       Lesotho  2023   3.186   4.548110
288       Lebanon  2023   2.707   4.682796
289   Afghanistan  2023   1.721   3.304115

[290 rows x 4 columns]


In [15]:
df3=df2[df2["Year"]==2023]
df4=df2[df2["Year"]==2024]

In [16]:
df3=df3.sort_values(by=["Predicted"],ascending=False)
df4=df4.sort_values(by=["Predicted"],ascending=False)
print(df3.head(10))
print(df4.head(10))

         Country  Year  Actual  Predicted
147      Finland  2023   7.741   7.411308
150       Sweden  2023   7.344   7.386824
148      Denmark  2023   7.583   7.383816
153       Norway  2023   7.302   7.375235
176    Singapore  2023   6.523   7.344286
155  Switzerland  2023   7.060   7.239696
154   Luxembourg  2023   7.122   7.226577
163      Ireland  2023   6.838   7.182736
157  New Zealand  2023   7.029   7.133396
149      Iceland  2023   7.525   7.123506
        Country  Year  Actual  Predicted
0       Finland  2024   7.736   7.795816
6        Norway  2024   7.262   7.779664
1       Denmark  2024   7.521   7.762057
3        Sweden  2024   7.345   7.759749
33    Singapore  2024   6.565   7.744543
12  Switzerland  2024   6.935   7.642823
8    Luxembourg  2024   7.122   7.639113
14      Ireland  2024   6.889   7.621478
2       Iceland  2024   7.515   7.614133
11  New Zealand  2024   6.952   7.467163


#Making predictions for 2025

##Since there is no actual data for predicting 2025 ranking, the model is using 2024 values.

In [17]:
x2025 = testdf[testdf['Year'] == 2024][features]  # extract features from the original test data
x2025 = imputer.transform(x2025)
x2025scaled = xscaler.transform(x2025)

In [18]:
# Inverse transform predictions to original scale and flatten to 1D
a=yscaler.inverse_transform(model2025.predict(x2025scaled).reshape(-1, 1)).flatten()

# Get country names for the context
countries = testdf[testdf['Year'] == 2024]['Country name'].values

# Create the DataFrame correctly using a dictionary for columns
p2025 = pd.DataFrame({
    'Country': countries,
    'Predicted Ladder Score 2025': a
})

In [19]:
p2025=p2025.sort_values(by=['Predicted Ladder Score 2025'],ascending=False)[['Country','Predicted Ladder Score 2025']]

In [20]:
p2025.head(10)

,Country,Predicted Ladder Score 2025
0,Finland,7.795816
6,Norway,7.779664
1,Denmark,7.762057
3,Sweden,7.759749
33,Singapore,7.744543
12,Switzerland,7.642823
8,Luxembourg,7.639113
14,Ireland,7.621478
2,Iceland,7.614133
11,New Zealand,7.467163


In [21]:
p2025['Previous Year Rank']=(p2025.index)+1
p2025top10=p2025.reset_index(drop=True).head(10)
p2025top10['Rank']=(p2025top10.index)+1
p2025top10.to_csv('pred2025.csv')

In [22]:
p2025top10

,Country,Predicted Ladder Score 2025,Previous Year Rank,Rank
0,Finland,7.795816,1,1
1,Norway,7.779664,7,2
2,Denmark,7.762057,2,3
3,Sweden,7.759749,4,4
4,Singapore,7.744543,34,5
5,Switzerland,7.642823,13,6
6,Luxembourg,7.639113,9,7
7,Ireland,7.621478,15,8
8,Iceland,7.614133,3,9
9,New Zealand,7.467163,12,10


#Checking r squared values

In [23]:
print("Train R2:", model2025.score(xtrainscaled, ytrainscaled))
print("Test R2:", model2025.score(xtestscaled, ytestscaled))

Train R2: 0.7175469091853337
Test R2: 0.5701592430032327
